In [40]:
from ML_pipeline.data_preprocessing import HeartbeatDataProcessor 

In [41]:
folder_path='../data/PAMAP2_Dataset/Protocol/'
filtered_df_path='../ML_pipeline/'
processed_data = HeartbeatDataProcessor(folder_path,filtered_df_path)
processed_data.preprocess_subjects(range(101,108))
# print(processed_data.df_filtered.head(3))
# print(processed_data.filtered_index.head(3))

Selected DataFrame columns have 0.2913% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.019% NaNs!

successfully loaded subject 101
Selected DataFrame columns have 0.4147% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.0327% NaNs!

successfully loaded subject 102
Selected DataFrame columns have 0.1624% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.0117% NaNs!

successfully loaded subject 103
Selected DataFrame columns have 0.3568% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.0095% NaNs!

successfully loaded subject 104
Selected DataFrame columns have 0.3409% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.0268% NaNs!

successfully loaded subject 105
Selected DataFrame columns have 0.2521% NaNs.
Interpolating selected columns...
Selected DataFrame columns now have 0.0349% NaNs!

successfully loaded subject 106
Selected DataFrame colu

In [42]:
print(processed_data.df_filtered.head(3))

       0  1   2        3        4        5         6        7        8  \
0  89.00  1 NaN  30.1875 -9.63930  2.40690  0.426408 -9.49891  2.41160   
1  89.01  1 NaN  30.1875 -9.29438  2.51910  0.546659 -9.49853  2.45691   
2  89.02  1 NaN  30.1875 -9.41256  2.40515  0.429810 -9.55869  2.47244   

          9  ...        46       47       48       49        50        51  \
0  0.526730  ...  0.023993 -51.3613 -45.8777 -2.12315  0.483022 -0.458425   
1  0.526630  ... -0.025650 -51.8734 -45.7230 -1.98330  0.482988 -0.458543   
2  0.526658  ...  0.004263 -51.4929 -45.6132 -1.98209  0.482891 -0.458596   

         52        53  subject_id  interval_id  
0  0.660054 -0.347655         107           78  
1  0.659992 -0.347665         107           78  
2  0.660006 -0.347705         107           78  

[3 rows x 56 columns]


## Random Forest on subject 101 segments

This builds per-window statistical features from `processed_data.subject_segment_dict[101]`, then trains and evaluates a Random Forest classifier.

In [45]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

slices = processed_data.subject_segment_dict[101]
for i in range(101,108):
    slices.extend(processed_data.subject_segment_dict[i])

feature_rows = []
labels = []
groups = []

for slice in slices:
    if slice.empty:
        continue

    # Column 1 is activity id in PAMAP2 raw files.
    label = int(slice[1].mode().iloc[0])

    # Keep numeric sensor columns, excluding metadata/label columns.
    drop_cols = [0, 1,2, 7,8,9,16,17,18,19,24,25,26,33,34,35,36,41,42,43,50,51,52,53, "subject_id", "interval_id"]
    sensor_df = slice.drop(columns=drop_cols, errors="ignore")
    sensor_df = sensor_df.select_dtypes(include=[np.number])

    if sensor_df.shape[1] == 0:
        continue

    row = {}
    for col in sensor_df.columns:
        values = sensor_df[col]
        row[f"c{col}_mean"] = values.mean()
        row[f"c{col}_std"] = values.std()
        row[f"c{col}_min"] = values.min()
        row[f"c{col}_max"] = values.max()

    feature_rows.append(row)
    labels.append(label)
    groups.append(int(slice["interval_id"].iloc[0]))


In [48]:


X = pd.DataFrame(feature_rows)
y = pd.Series(labels, name="activity_id")
groups = pd.Series(groups, name="interval_id")

# print(f"X shape: {X.shape}")
# print("Class counts:")
# print(y.value_counts().sort_index())

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

rf_pipeline = Pipeline([
    # ("imputer", SimpleImputer(strategy="median")),
    (
        "rf",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            max_depth=24,
            n_jobs=-1,
            class_weight="balanced_subsample",
        ),
    ),
])

rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels_sorted = np.sort(np.unique(np.concatenate([y_test.to_numpy(), y_pred])))
cm = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels_sorted),
    index=[f"true_{lbl}" for lbl in labels_sorted],
    columns=[f"pred_{lbl}" for lbl in labels_sorted],
)

print("\nConfusion matrix (test set):")
print(cm)


Accuracy: 0.8113

Classification report:
              precision    recall  f1-score   support

           1       1.00      0.87      0.93       946
           2       0.77      0.97      0.86       462
           3       0.00      0.00      0.00         0
           4       0.00      0.00      0.00         0
           5       1.00      0.98      0.99       482
           6       0.85      0.23      0.36       234
           7       1.00      0.62      0.77       919
          12       0.91      0.98      0.94       198
          13       0.84      0.99      0.91        99
          16       0.81      0.87      0.84       863
          17       0.00      0.00      0.00         0
          24       0.00      0.00      0.00         0

    accuracy                           0.81      4203
   macro avg       0.60      0.54      0.55      4203
weighted avg       0.92      0.81      0.84      4203


Confusion matrix (test set):
         pred_1  pred_2  pred_3  pred_4  pred_5  pred_6  pred